# 42 — Threshold analysis: choosing the operating point

**Reads:** `ml.train_matrix`, `ml.test_matrix`, and the registered `@champion` model

Ported from the threshold sections of `05_modelling`. A trained model outputs a probability;
a clinical service needs a *decision*. The threshold is what turns one into the other — and
**it is a policy choice, not a modelling one.**

That distinction matters more than it sounds. Nothing in the training data tells you whether
missing a poor outcome is worse than raising a false alarm. That is a judgement about
clinical capacity, patient harm, and what happens to a flagged patient — and it belongs to
the service, not to the person who fitted the model.

What this notebook does is make the trade-off legible enough for the service to choose:
here is what you get at each threshold, in patients rather than percentages.

## 1. Load the model and the frozen split

In [ ]:
import mlflow, mlflow.sklearn
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from mlflow import MlflowClient

EXPERIMENT       = "knee-poor-outcome"
REGISTERED_MODEL = "knee-poor-outcome-ebm"
CHAMPION_LABEL   = "champion"   # a version tag, not an MLflow alias -- Fabric has no alias API

# --- champion resolution ----------------------------------------------------
# Fabric's MLflow registry does not implement the alias endpoints
# (/api/2.0/mlflow/registered-models/alias -> 404), so `@champion` URIs and
# get_model_version_by_alias() are unavailable. Version tags work, so the
# champion pointer is the tag champion='true'. This helper tries the alias
# first so the code stays correct on a tenant where aliases do exist.
def resolve_champion(client, name):
    try:
        return client.get_model_version_by_alias(name, "champion")
    except Exception:
        pass
    versions = list(client.search_model_versions(f"name='{name}'"))
    if not versions:
        raise RuntimeError(f"No versions registered for {name} -- run 40_train_register first.")
    champions = [v for v in versions if v.tags.get("champion") == "true"]
    if champions:
        return max(champions, key=lambda v: int(v.version))
    latest = max(versions, key=lambda v: int(v.version))
    print(f"WARNING: no champion tag found; falling back to latest version {latest.version}")
    return latest


mlflow.set_experiment(EXPERIMENT)
mlflow.autolog(disable=True)

def load(table):
    pdf = spark.table(table).toPandas()
    y = pdf.pop("outcome_label").astype(int)
    return pdf, y

X_train, y_train = load("ml.train_matrix")
X_test,  y_test  = load("ml.test_matrix")
X_test = X_test[X_train.columns]

mv = resolve_champion(MlflowClient(), REGISTERED_MODEL)
model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL}/{mv.version}")
print(f"{REGISTERED_MODEL} v{mv.version} (champion)")

y_prob = model.predict_proba(X_test)[:, 1]
PREVALENCE = float(y_test.mean())
print(f"test patients {len(y_test):,} | poor outcomes {int(y_test.sum()):,} ({PREVALENCE:.1%})")

## 2. The threshold sweep, in patients

Percentages hide the thing that matters. A clinic manager does not act on "recall 0.34"; they
act on "you will flag 1,900 patients a year and be right about 1,100 of them, and 3,600 poor
outcomes will slip past".

Every column below is a count.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

def sweep(y_true, y_prob, thresholds):
    rows = []
    for t in thresholds:
        y_hat = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
        rows.append({
            "threshold":        t,
            "flagged":          int(tp + fp),
            "correctly_flagged": int(tp),
            "false_alarms":     int(fp),
            "missed":           int(fn),
            "precision":        round(float(precision_score(y_true, y_hat, zero_division=0)), 3),
            "recall":           round(float(recall_score(y_true, y_hat, zero_division=0)), 3),
            "f1":               round(float(f1_score(y_true, y_hat, zero_division=0)), 3),
            "pct_of_cohort_flagged": round((tp + fp) / len(y_true) * 100, 1),
        })
    return pd.DataFrame(rows)

table = sweep(y_test, y_prob, [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
display(table)

print("Read the two right-hand columns together: 'false_alarms' is patients given an "
      "unnecessary conversation; 'missed' is patients who get a disappointing operation "
      "nobody warned them about. Neither is free.")

## 3. The 80%-precision operating point

The original analysis targeted 80% precision: *four out of five flagged patients should
genuinely be at risk*, on the reasoning that a pre-operative optimisation pathway loses
clinical credibility below that.

That is a defensible starting point and it is worth stating as a choice rather than a
constant. What it costs in recall is the number below.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

def threshold_for_precision(target):
    ok = precision[:-1] >= target
    if not ok.any():
        return None, 0.0, 0.0
    i = int(np.argmax(recall[:-1] * ok))
    return float(thresholds[i]), float(precision[i]), float(recall[i])

print(f"{'target':>8} {'threshold':>10} {'precision':>10} {'recall':>8}   found / missed")
print("-" * 62)
operating_points = {}
for target in [0.60, 0.70, 0.80, 0.90]:
    t, p, r = threshold_for_precision(target)
    if t is None:
        print(f"{target:>8.0%} {'unreachable':>10}")
        continue
    operating_points[target] = t
    found = int(r * y_test.sum())
    print(f"{target:>8.0%} {t:>10.3f} {p:>10.3f} {r:>8.3f}   "
          f"{found:,} / {int(y_test.sum()) - found:,}")

THRESHOLD_P80 = operating_points.get(0.80)
print(f"\nAP: {average_precision_score(y_test, y_prob):.4f} "
      f"(no-skill {PREVALENCE:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.plot(recall, precision, lw=2.5, color="#4C78A8", label="Calibrated EBM")
ax.axhline(PREVALENCE, color="#888", ls="--", lw=1.5, label=f"no skill ({PREVALENCE:.2f})")

for target, colour in [(0.60, "#72B7B2"), (0.80, "#C44E52"), (0.90, "#8B4A9C")]:
    t, p, r = threshold_for_precision(target)
    if t is None:
        continue
    ax.plot(r, p, "o", ms=10, color=colour, zorder=5)
    ax.annotate(f"P={target:.0%}\nthr {t:.2f}\nrecall {r:.2f}",
                xy=(r, p), xytext=(r + .06, p + .05),
                fontsize=9, color=colour, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=colour, lw=1.2))

ax.set_xlabel("Recall — share of poor outcomes found")
ax.set_ylabel("Precision — share of flags that are correct")
ax.set_title("Every point on this curve is a different clinical policy")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc="upper right"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4. Train / validation / test at the chosen threshold

Three numbers, not one. A large train-to-test gap means the model memorised; a large
validation-to-test gap means the split was unlucky. Both are things to know before the
threshold is written into a care pathway.

Validation figures come from 5-fold `cross_val_predict` on the training set, so no patient
contributes to their own prediction.

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.base import clone

THRESHOLDS_TO_REPORT = [0.5, THRESHOLD_P80, 0.8]
THRESHOLDS_TO_REPORT = sorted({round(t, 3) for t in THRESHOLDS_TO_REPORT if t is not None})

print("Computing out-of-fold validation predictions (this refits the model 5 times)...")
prob_val = cross_val_predict(
    clone(model), X_train, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method="predict_proba", n_jobs=-1)[:, 1]
prob_train = model.predict_proba(X_train)[:, 1]

rows = []
for t in THRESHOLDS_TO_REPORT:
    for split, yt, yp in [("train", y_train, prob_train),
                          ("validation", y_train, prob_val),
                          ("test", y_test, y_prob)]:
        y_hat = (yp >= t).astype(int)
        rows.append({
            "threshold": t, "split": split,
            "precision": round(float(precision_score(yt, y_hat, zero_division=0)), 3),
            "recall":    round(float(recall_score(yt, y_hat, zero_division=0)), 3),
            "f1":        round(float(f1_score(yt, y_hat, zero_division=0)), 3),
            "flagged":   int(y_hat.sum()),
        })

summary = pd.DataFrame(rows).pivot(index=["threshold", "split"],
                                   columns=[], values=["precision", "recall", "f1", "flagged"])
display(pd.DataFrame(rows).set_index(["threshold", "split"]))

gap = pd.DataFrame(rows).pivot_table(index="threshold", columns="split", values="recall")
gap["train_minus_test"] = (gap["train"] - gap["test"]).round(3)
print("\nRecall gap, train vs test (large = memorisation):")
print(gap[["train", "validation", "test", "train_minus_test"]].to_string())

## 5. Confusion matrices — what the threshold does to real people

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

n = len(THRESHOLDS_TO_REPORT)
fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4.2))
axes = np.atleast_1d(axes)

for ax, t in zip(axes, THRESHOLDS_TO_REPORT):
    y_hat = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_hat, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Good", "Poor"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=",")
    tn, fp, fn, tp = cm.ravel()
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    ax.set_title(f"threshold {t}\nprecision {prec:.2f} · recall {rec:.2f}", fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

plt.tight_layout(); plt.show()

print("Bottom-left is the cell that matters clinically: poor outcomes the model did not flag.")

## 6. What each threshold means in practice

Adapted from the original notebook's framing, which was good and is worth keeping verbatim
in spirit: describe the *service*, not the matrix.

In [ ]:
ANNUAL_KNEE_REPLACEMENTS = 100_000     # roughly the NHS England annual figure

print("Scaled to a national programme of "
      f"{ANNUAL_KNEE_REPLACEMENTS:,} knee replacements a year:\n")

scaled = []
for t in [0.3, 0.5, THRESHOLD_P80, 0.8]:
    if t is None:
        continue
    y_hat = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_hat, labels=[0, 1]).ravel()
    k = ANNUAL_KNEE_REPLACEMENTS / len(y_test)
    scaled.append({
        "threshold":            round(t, 3),
        "patients_flagged":     f"{int((tp + fp) * k):,}",
        "correctly_flagged":    f"{int(tp * k):,}",
        "unnecessary_reviews":  f"{int(fp * k):,}",
        "poor_outcomes_missed": f"{int(fn * k):,}",
    })
display(pd.DataFrame(scaled))

print("""
Reading the ends of the range:

  LOW threshold  — flag widely. Catch most poor outcomes; spend a lot of clinic time on
                   patients who would have done fine. Viable only if what a flag triggers
                   is cheap and harmless, like a leaflet and a conversation.

  HIGH threshold — flag rarely. Almost every flag is real; most poor outcomes go unflagged.
                   Appropriate when a flag triggers something expensive or invasive, such
                   as delaying surgery for a pre-habilitation programme.

The right answer depends entirely on what happens to a flagged patient. Ask the clinical
team that question before picking a number — and put their answer in the model card.
""")

## 7. Does the threshold hold across subgroups?

A threshold tuned on the whole cohort can behave very differently inside it. If precision at
the chosen cut-off collapses for the over-80s, the service is applying a rule that works for
most patients and quietly fails for one group.

Under the EU AI Act this is not an optional extra — it is the bias and fairness testing that
high-risk systems have to evidence. Any group where precision drops materially belongs in
the model card as a documented limitation.

In [ ]:
raw_test = spark.table("ml.test_raw").toPandas()
if len(raw_test) != len(y_test):
    print(f"WARNING: ml.test_raw has {len(raw_test):,} rows, expected {len(y_test):,} — "
          "re-run 40_train_register §5b so the tables agree.")

t = THRESHOLD_P80 or 0.5
y_hat = (y_prob >= t).astype(int)

def subgroup_report(series, label):
    out = []
    for value in sorted(series.dropna().unique()):
        m = (series == value).values
        if m.sum() < 100:
            continue
        yt, yh = np.asarray(y_test)[m], y_hat[m]
        if yh.sum() == 0:
            out.append({label: value, "n": int(m.sum()), "flagged": 0,
                        "precision": None, "recall": 0.0,
                        "actual_poor_rate": round(float(yt.mean()), 3)})
            continue
        out.append({
            label: value, "n": int(m.sum()), "flagged": int(yh.sum()),
            "precision": round(float(precision_score(yt, yh, zero_division=0)), 3),
            "recall":    round(float(recall_score(yt, yh, zero_division=0)), 3),
            "actual_poor_rate": round(float(yt.mean()), 3),
        })
    return pd.DataFrame(out)

print(f"Subgroup performance at threshold {t:.3f}\n")
for col, label in [("age_band_grouped", "age band"), ("sex", "sex"), ("region", "region")]:
    if col in raw_test.columns:
        rep = subgroup_report(raw_test[col], label)
        if len(rep):
            print(f"--- by {label} ---")
            print(rep.to_string(index=False)); print()

print("Look for: a group where precision falls well below the target, or where recall is "
      "near zero. Both are findings for the model card, not bugs to tune away.")
print("\nThe not_recorded rows are patients whose demographics NHS suppressed. They are a "
      "real subgroup, not a data-quality artefact -- report them, do not hide them.")

## 7b. Decision curve analysis — is this model worth using at all?

Every argument about precision versus recall is really an argument about the relative harm
of two mistakes, and neither metric states that harm openly. Decision curve analysis does.

**The idea** (Vickers & Elkin, 2006; recommended by TRIPOD): the risk threshold at which a
clinician would act *is* their value judgement, made explicit. A clinician who reviews
patients at 20% risk is saying they would accept **4 unnecessary reviews to catch 1 real
case**. That exchange rate falls straight out of the threshold:

```
harm ratio w = p / (1 - p)
```

| Threshold | Exchange rate | The clinician is saying |
|---|---|---|
| 0.10 | 1 : 9 | "review 9 unnecessarily to catch 1" — a cheap intervention |
| 0.20 | 1 : 4 | "review 4 unnecessarily to catch 1" |
| 0.50 | 1 : 1 | "a false alarm is as bad as a missed case" |
| 0.80 | 4 : 1 | "a false alarm is 4x worse than a missed case" |

**Net benefit** then puts both mistakes in one number, in units of true positives per patient:

```
NB = TP/N  -  (FP/N) x w
```

and gets compared against the two strategies that need no model at all:

* **review everyone** — catches every case, maximum false alarms
* **review no one** — current practice, net benefit 0 by definition

A model is worth deploying only over the range of thresholds where its curve is above both.
If it never is, it does not matter how good the AUC was.

**For this pipeline the "intervention" is a conversation, not an operation.** A flag is a
suggestion to a clinician — it does not defer or deny surgery. That makes a false positive
cheap (a consultation, some anxiety, some clinician time) and a false negative expensive
(a patient goes to theatre unprepared and nobody warned them). So the thresholds worth
examining are the **low** ones, and the curve below shows whether the model helps there.

In [ ]:
def net_benefit_table(y_true, y_prob, thresholds):
    """Net benefit per Vickers & Elkin, for the model and both no-model strategies."""
    y_true = np.asarray(y_true)
    n = len(y_true)
    prevalence = y_true.mean()
    rows = []
    for pt in thresholds:
        w = pt / (1 - pt)                      # exchange rate: FPs per TP
        y_hat = (y_prob >= pt).astype(int)
        tp = int(((y_hat == 1) & (y_true == 1)).sum())
        fp = int(((y_hat == 1) & (y_true == 0)).sum())
        rows.append({
            "threshold":      round(pt, 3),
            "harm_ratio_w":   round(w, 2),
            "flagged":        tp + fp,
            "nb_model":       tp / n - (fp / n) * w,
            "nb_review_all":  prevalence - (1 - prevalence) * w,
            "nb_review_none": 0.0,
        })
    df = pd.DataFrame(rows)
    # The interpretable form: how many patients per 1,000 do we correctly identify,
    # net of the false alarms, compared with reviewing nobody?
    df["net_per_1000"] = (df["nb_model"] * 1000).round(1)
    df["best_strategy"] = df.apply(
        lambda r: max([("model", r.nb_model), ("review all", r.nb_review_all),
                       ("review none", r.nb_review_none)], key=lambda t: t[1])[0], axis=1)
    return df

grid = np.round(np.arange(0.05, 0.85, 0.05), 3)
dca = net_benefit_table(y_test, y_prob, grid)
display(dca[["threshold", "harm_ratio_w", "flagged", "net_per_1000", "best_strategy"]])

useful = dca[dca["best_strategy"] == "model"]
if len(useful):
    print(f"The model beats both no-model strategies for thresholds "
          f"{useful['threshold'].min():.2f} to {useful['threshold'].max():.2f} "
          f"(harm ratios {useful['harm_ratio_w'].min():.1f} to {useful['harm_ratio_w'].max():.1f}).")
    best = useful.loc[useful["nb_model"].idxmax()]
    print(f"Peak net benefit at threshold {best['threshold']:.2f}: "
          f"{best['net_per_1000']:.1f} net patients correctly identified per 1,000 screened.")
else:
    print("The model never beats reviewing everyone or reviewing nobody. "
          "That is a finding -- it would mean this model should not be deployed.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(dca["threshold"], dca["nb_model"], lw=2.5, color="#4C78A8", label="Use the model")
ax.plot(dca["threshold"], dca["nb_review_all"], lw=1.8, ls="--", color="#C44E52",
        label="Review every patient")
ax.axhline(0, lw=1.8, ls=":", color="#555", label="Review no one (current practice)")

ax.fill_between(dca["threshold"], dca["nb_model"],
                np.maximum(dca["nb_review_all"], 0),
                where=(dca["nb_model"] > np.maximum(dca["nb_review_all"], 0)),
                alpha=.18, color="#4C78A8")

if THRESHOLD_P80:
    ax.axvline(THRESHOLD_P80, color="#8A5A12", lw=1.5, ls="-.",
               label=f"80%-precision point ({THRESHOLD_P80:.2f})")

lo = max(dca["nb_model"].min(), -dca["nb_model"].max())
ax.set_ylim(lo, dca["nb_model"].max() * 1.25)
ax.set_xlabel("Threshold probability — the risk at which a clinician would act")
ax.set_ylabel("Net benefit (true positives per patient, false positives discounted)")
ax.set_title("Decision curve: where does this model earn its place?", fontweight="bold")
ax.legend(loc="upper right"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print("""
How to read it:

  Where the blue line is highest, using the model beats both alternatives.
  Where the red dashed line is highest, just review everyone -- the model adds nothing.
  Where zero is highest, review no one; the model would do net harm.

The x-axis is not a modelling parameter. It is the clinical team's answer to
"how many unnecessary reviews would you accept to catch one patient who will not
benefit?" -- and the curve tells you whether the model helps at their answer.
""")

## 7c. Precision at capacity, and number needed to evaluate

The decision curve answers *"is the model worth using?"*. This answers *"what will next
Tuesday look like?"*, and it is the version a service manager can act on.

A pre-operative optimisation pathway has finite slots. So rather than picking a probability
threshold and seeing how many patients fall out, fix the capacity and ask how well the
model fills it: **take the N highest-risk patients — how many genuinely will not benefit?**

Two properties make this the most honest precision-style metric available:

* **It cannot be gamed.** Raw precision rewards a model that flags almost nobody; here every
  model is given exactly N slots, so caution buys nothing. It compares *ranking quality* at
  the point of use.
* **It converts directly into workload.** `NNE = 1 / precision` — the **number needed to
  evaluate**, or how many patients a clinician reviews to find one who genuinely will not
  benefit. That is the standard framing for clinical alerting systems, and it is a sentence
  a clinical director can answer yes or no to.

Set `CAPACITY_PER_1000` from what the service could actually absorb. The defaults below are
placeholders — **replace them with a real figure before this reaches a slide**, because an
invented denominator is worse than none.

In [ ]:
def precision_at_capacity(y_true, y_prob, k):
    """Precision among the k highest-risk patients. Uses ranking, ignores the threshold."""
    y_true = np.asarray(y_true)
    k = min(int(k), len(y_true))
    top = np.argsort(y_prob)[::-1][:k]
    hits = int(y_true[top].sum())
    return hits, k, hits / k if k else 0.0

# Capacity expressed per 1,000 patients screened, so it scales to any cohort size.
# TODO: replace with the real figure from the clinical team.
CAPACITY_PER_1000 = [20, 50, 100, 200]     # i.e. 2%, 5%, 10%, 20% of those screened

n_test = len(y_test)
rows = []
for per_1000 in CAPACITY_PER_1000:
    k = int(round(n_test * per_1000 / 1000))
    hits, k_used, prec = precision_at_capacity(y_test, y_prob, k)
    cutoff = float(np.sort(y_prob)[::-1][min(k_used, n_test) - 1])
    rows.append({
        "reviews_per_1000":   per_1000,
        "patients_reviewed":  k_used,
        "genuinely_at_risk":  hits,
        "precision_at_k":     round(prec, 3),
        "NNE":                round(1 / prec, 1) if prec else None,
        "implied_threshold":  round(cutoff, 3),
        "share_of_all_cases": round(hits / int(np.sum(y_test)), 3),
    })

cap = pd.DataFrame(rows)
display(cap)

base = float(np.mean(y_test))
print(f"Reviewing patients at random would give precision {base:.3f}  (NNE {1/base:.1f})")
print()
for _, r in cap.iterrows():
    lift = r["precision_at_k"] / base
    print(f"  Review the top {r['reviews_per_1000']:>3} per 1,000: "
          f"{r['genuinely_at_risk']:>4} of {r['patients_reviewed']:>4} genuinely at risk "
          f"(NNE {r['NNE']}, {lift:.1f}x better than random), "
          f"catching {r['share_of_all_cases']:.0%} of all poor outcomes")

print("""
The sentence to take to a clinical director:

  "If you can review N patients a month, the model fills those slots with X patients
   who genuinely will not benefit -- you review NNE patients to find each one, against
   RANDOM if you picked at random. You will catch Y% of all poor outcomes."

Note the tension this makes explicit and unavoidable: raising capacity always lowers
precision and always raises coverage. There is no threshold that escapes it -- only a
choice about where on that trade-off the service wants to sit.""")

## 8. Record the decision

The chosen threshold is a governed artefact. It is logged to MLflow and tagged onto the model
version, so the number the app uses can always be traced to the analysis that produced it.

In [ ]:
# THRESHOLD_P80 is None whenever no threshold reaches 80% precision on this test set.
# That is not a bug to paper over -- it is the model telling you 80% is not on offer.
# Fall back down the target ladder and say so out loud, rather than dying mid-demo.
if THRESHOLD_P80 is not None:
    CHOSEN_THRESHOLD = THRESHOLD_P80
    CHOSEN_TARGET    = 0.80
else:
    reachable = sorted(operating_points.items(), reverse=True)
    if not reachable:
        raise RuntimeError(
            "No threshold reaches even 60% precision. Do not pick an operating point "
            "from this model -- go back to 41_model_search, or argue from the decision "
            "curve in section 7b that the model should not be deployed at all.")
    CHOSEN_TARGET, CHOSEN_THRESHOLD = reachable[0]
    print(f"WARNING: 80% precision is unreachable on this test set. "
          f"Falling back to the highest reachable target, {CHOSEN_TARGET:.0%}.")
    print("Say this on stage rather than hiding it -- 'the data would not support the "
          "precision we wanted' is a more honest slide than a threshold nobody can hit.")

CHOSEN_RATIONALE = (
    f"Highest recall achievable while holding precision at {CHOSEN_TARGET:.0%}. "
    "Rationale: a pre-operative optimisation pathway loses clinical credibility "
    "when a large share of its flags are wrong. The target was set with the "
    "clinical team, cross-checked against the decision curve (7b) and against "
    "review capacity (7c); it is a service decision, not a modelling one.")

CHOSEN_THRESHOLD = float(CHOSEN_THRESHOLD)

with mlflow.start_run(run_name="threshold-analysis") as run:
    mlflow.set_tags({
        "run_type":      "threshold_analysis",
        "notebook":      "42_threshold_analysis",
        "model_name":    REGISTERED_MODEL,
        "model_version": str(mv.version),
        "risk_class":    "high_risk_eu_ai_act",
    })
    mlflow.log_param("chosen_threshold",  round(CHOSEN_THRESHOLD, 4))
    mlflow.log_param("precision_target",  CHOSEN_TARGET)
    mlflow.log_param("rationale",         CHOSEN_RATIONALE)

    y_hat = (y_prob >= CHOSEN_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_hat, labels=[0, 1]).ravel()
    mlflow.log_metrics({
        "chosen_precision": float(precision_score(y_test, y_hat, zero_division=0)),
        "chosen_recall":    float(recall_score(y_test, y_hat, zero_division=0)),
        "flagged":          int(tp + fp),
        "false_alarms":     int(fp),
        "missed":           int(fn),
    })
    mlflow.log_table(table, "threshold_sweep.json")

client = MlflowClient()
client.set_model_version_tag(
    REGISTERED_MODEL, mv.version, "operating_threshold", f"{CHOSEN_THRESHOLD:.4f}")
client.set_model_version_tag(
    REGISTERED_MODEL, mv.version, "operating_threshold_target", f"{CHOSEN_TARGET:.2f}")
client.set_model_version_tag(
    REGISTERED_MODEL, mv.version, "operating_threshold_rationale", CHOSEN_RATIONALE)

print(f"\nThreshold {CHOSEN_THRESHOLD:.4f} (target precision {CHOSEN_TARGET:.0%}) "
      f"recorded on {REGISTERED_MODEL} v{mv.version}.")
print(f"At that threshold: {tp + fp:,} flagged, {tp:,} genuinely at risk, "
      f"{fp:,} false alarms, {fn:,} missed.")
print("The risk bands in 50_batch_score are a separate clinical policy \u2014 if this "
      "threshold moves, revisit them too.")


---

**Next:** `43_explainability` — the glassbox shape functions and the black-box contrast.